# xPass Model & Everton Case Study - Technical Task Submission

**Competition Focus:** Top 5 European Leagues - 2015/16 Season  
**Dataset:** StatsBomb Open Data (free event data repository)  
**Purpose:** Build, calibrate, and apply an **Expected Passes (xP) model** to quantify passing difficulty and decision-making, aligned with a practical football club decision-making context. This work is submitted as part of a Data Scientist Technical Task, focusing on actionable insight rather than purely descriptive analysis.  
**Question:** How can player and team passing performance be evaluated probabilistically to identify risk-taking, conservative, and high-impact passers?  
**Audience:** Football analysts & coaching staff at a professional club.  
**Decision Impact:** Inform coaching and tactical decisions around player usage & risk management.  
**Methods:** Feature engineering, probabilistic modelling, model calibration, performance validation, and tactical application using Everton as a case study.  
**Author:** [Victoria Friss de Kereki](https://www.linkedin.com/in/victoria-friss-de-kereki/)  
**Medium Articles:**  
[Beyond Pass Completion: Building an Expected Pass Model in Football](https://medium.com/@vickyfrissdekereki/beyond-pass-completion-building-an-expected-pass-model-in-football-c02d100d667a)   
[What Pass Clustering Visualisations Reveal About Football Team Playing Style](https://medium.com/@vickyfrissdekereki/what-pass-clustering-visualisations-reveal-about-football-team-playing-style-012cb2cab776)   

**Notebook first written:** `04/04/2026`  
**Last updated:** `13/04/2026`

> This notebook develops and validates an Expected Passes (xP) model to estimate the likelihood of pass completion in context, using event data from the 2015/16 season across Europe’s top five leagues. The model is designed to produce interpretable, actionable outputs suitable for football club decision-making.  
>
> Beyond model development, the notebook demonstrates practical application by analysing Everton’s passing profiles, identifying extremely safe passers and players struggling with difficult passes. Insights are framed with club decisions in mind, offering tactical recommendations and player evaluation frameworks.  
>
> This submission adheres to the technical task brief by producing focused, interpretable analysis intended to influence real-world football decisions.  

  
---------------------

## 1. Packages and Configuration

The project uses Python packages for data manipulation, visualisation, football analytics, and modelling. Warnings are suppressed to maintain clean output.
- **Data manipulation:** `pandas`, `numpy`
- **Visualisation:** `matplotlib`, `seaborn`, `mplsoccer`
- **Football analytics:** `statsbombpy`
- **Machine learning:** `sklearn`
- **Utilities:** `tqdm`, `pathlib`

In [1]:
# Standard library
import math
import re
import warnings
from pathlib import Path
import pprint

# Data manipulation
import numpy as np
import pandas as pd

# Plotting and visualisation
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import matplotlib.patheffects as path_effects
from matplotlib.colors import LinearSegmentedColormap, Normalize, to_rgb
from matplotlib.cm import ScalarMappable
from matplotlib.patches import FancyArrowPatch
import seaborn as sns
from adjustText import adjust_text

# Football analytics libraries
from mplsoccer import Pitch, VerticalPitch
from highlight_text import ax_text, fig_text
from statsbombpy import sb

# Machine learning and modelling
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import roc_auc_score, brier_score_loss

# Utilities
from tqdm import tqdm

warnings.filterwarnings("ignore", message="credentials were not supplied")

## 2. Load Competition, Match & Event Data

The project begins by exploring the competitions available in the StatsBomb open dataset. The focus is on male top-flight leagues in the 2015/16 season.  

Steps:
1. Filter competitions to select the top five European leagues.
2. Count the number of matches available in each competition, to work with complete (or almost complete) seasons only.
3. Prepare to extract event-level data, focusing on passes only.

### 2.1 Download Matches for La Liga (2015/16)

In [2]:
leagues = [(11, 27)]  # La Liga

all_matches = []

for comp_id, season_id in leagues:
    matches = sb.matches(
        competition_id=comp_id,
        season_id=season_id
    )
    
    # keep league identifier for later analysis
    matches["competition_id"] = comp_id
    
    all_matches.append(matches)

matches_df = pd.concat(all_matches, ignore_index=True)

In [3]:
len(matches_df)

380

### 2.2 Extract Pass Events

Event data for the selected matches is extracted, keeping only pass events.  

Key steps:
- Use caching to avoid repeated downloads.
- Retain match IDs for linking events with match-level metadata.
- Combine all pass events into a single dataset.
- Inspect play patterns to ensure analysis focuses on relevant phases (e.g., Regular Play, Kick Off, Throw In).

In [4]:
# Cache folder
CACHE_DIR = Path("statsbomb_cache/events_15_16")
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def load_events(match_id):
    """Load events for a match, using cache if available."""
    cache_file = CACHE_DIR / f"{match_id}.parquet"

    if cache_file.exists():
        return pd.read_parquet(cache_file)

    events = sb.events(match_id=match_id)
    events.to_parquet(cache_file)
    return events

In [5]:
all_passes = []

for match_id in tqdm(matches_df["match_id"], desc="Extracting passes"):
    events = load_events(match_id)
    passes = events[events["type"] == "Pass"].copy()
    passes["match_id"] = match_id
    all_passes.append(passes)

passes_df = pd.concat(all_passes, ignore_index=True)

Extracting passes: 100%|██████████| 380/380 [00:08<00:00, 46.58it/s]


In [6]:
len(passes_df)

365288

In [7]:
passes_clean = passes_df.dropna(axis=1, how="all") # remove empty columns (those that do not have anything to do with passes)
passes_clean.head()

,counterpress,duration,id,index,location,match_id,minute,off_camera,out,pass_aerial_won,...,possession,possession_team,possession_team_id,related_events,second,team,team_id,timestamp,type,under_pressure
0,None,0.486053,31ad200d-972c-4a06-ad7c-152e65b8578c,5,"[61.0, 40.1]",3825848,0,True,None,None,...,2,Eibar,322,[0fb170d0-078b-4eec-bd0f-93535060bf53],0,Eibar,322,00:00:00.500,Pass,None
1,None,1.397421,0318204e-f23e-4493-ba61-f1ab39499285,7,"[61.2, 41.3]",3825848,0,True,None,None,...,2,Eibar,322,[03874a60-1451-44b3-9403-ede0dd55dad3],0,Eibar,322,00:00:00.986,Pass,None
2,None,1.558912,dd445101-ad1b-4de8-8ffc-b5fd3d1f976a,9,"[49.2, 33.5]",3825848,0,None,None,None,...,2,Eibar,322,[a4951807-804c-4166-9328-c636d5beed2e],4,Eibar,322,00:00:04.078,Pass,None
3,None,1.633776,01a2c6b0-b72b-4771-aa79-fca845a41857,12,"[32.3, 58.4]",3825848,0,None,None,None,...,2,Eibar,322,[43c4d791-3f73-4620-a406-139d56b2af1f],7,Eibar,322,00:00:07.214,Pass,None
4,None,2.934427,5cd78c17-e82d-454d-b76e-3286edb1936d,15,"[40.5, 76.1]",3825848,0,None,None,None,...,2,Eibar,322,[50ac42fc-bba4-4ac7-9110-2dc620a465bf],13,Eibar,322,00:00:13.855,Pass,None


In [8]:
def extract_coords(coord_col):
    # Convert everything to string first, then split
    x = pd.to_numeric(coord_col.astype(str).str.strip("[]").str.split().str[0], errors="coerce")
    y = pd.to_numeric(coord_col.astype(str).str.strip("[]").str.split().str[1], errors="coerce")
    return x, y

# Apply to location columns
passes_clean["start_x"], passes_clean["start_y"] = extract_coords(passes_clean["location"])
passes_clean["end_x"], passes_clean["end_y"] = extract_coords(passes_clean["pass_end_location"])

# Check missing values
cols = ["start_x", "start_y", "end_x", "end_y"]
print(passes_clean[cols].isna().sum())

C:\Users\vicky\AppData\Local\Temp\ipykernel_33752\3133970722.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passes_clean["start_x"], passes_clean["start_y"] = extract_coords(passes_clean["location"])
C:\Users\vicky\AppData\Local\Temp\ipykernel_33752\3133970722.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passes_clean["start_x"], passes_clean["start_y"] = extract_coords(passes_clean["location"])


start_x    0
start_y    0
end_x      0
end_y      0
dtype: int64


C:\Users\vicky\AppData\Local\Temp\ipykernel_33752\3133970722.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passes_clean["end_x"], passes_clean["end_y"] = extract_coords(passes_clean["pass_end_location"])
C:\Users\vicky\AppData\Local\Temp\ipykernel_33752\3133970722.py:9: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  passes_clean["end_x"], passes_clean["end_y"] = extract_coords(passes_clean["pass_end_location"])


In [9]:
# Remove Injury Clearance passes
passes_clean = passes_clean[passes_clean["pass_outcome"] != "Injury Clearance"].copy()

# Quick check
passes_clean["pass_outcome"].value_counts(dropna=False)

pass_outcome
None            276763
Incomplete       76191
Out               7601
Unknown           2465
Pass Offside      1720
Name: count, dtype: int64

In [10]:
PITCH_LENGTH = 120
PITCH_WIDTH = 80

# Start location checks
start_outside = passes_clean[
    (passes_clean["start_x"] <= 0) | (passes_clean["start_x"] > PITCH_LENGTH) |
    (passes_clean["start_y"] <= 0) | (passes_clean["start_y"] > PITCH_WIDTH)
]

# End (reception) location checks
end_outside = passes_clean[
    (passes_clean["end_x"] <= 0) | (passes_clean["end_x"] > PITCH_LENGTH) |
    (passes_clean["end_y"] <= 0) | (passes_clean["end_y"] > PITCH_WIDTH)
]

print(f"Start locations outside pitch: {len(start_outside)}")
print(f"End locations outside pitch: {len(end_outside)}")

Start locations outside pitch: 1
End locations outside pitch: 15


In [11]:
# Define pitch limits
PITCH_LENGTH = 120
PITCH_WIDTH = 80

# Fix start coordinates
passes_clean.loc[passes_clean["start_x"] > PITCH_LENGTH, "start_x"] = PITCH_LENGTH
passes_clean.loc[passes_clean["start_x"] < 0, "start_x"] = 0

passes_clean.loc[passes_clean["start_y"] > PITCH_WIDTH, "start_y"] = PITCH_WIDTH
passes_clean.loc[passes_clean["start_y"] < 0, "start_y"] = 0

# Fix end coordinates
passes_clean.loc[passes_clean["end_x"] > PITCH_LENGTH, "end_x"] = PITCH_LENGTH
passes_clean.loc[passes_clean["end_x"] < 0, "end_x"] = 0

passes_clean.loc[passes_clean["end_y"] > PITCH_WIDTH, "end_y"] = PITCH_WIDTH
passes_clean.loc[passes_clean["end_y"] < 0, "end_y"] = 0

# Quick check for remaining out-of-bounds
start_outside = passes_clean[
    (passes_clean["start_x"] < 0) | (passes_clean["start_x"] > PITCH_LENGTH) |
    (passes_clean["start_y"] < 0) | (passes_clean["start_y"] > PITCH_WIDTH)
]
end_outside = passes_clean[
    (passes_clean["end_x"] < 0) | (passes_clean["end_x"] > PITCH_LENGTH) |
    (passes_clean["end_y"] < 0) | (passes_clean["end_y"] > PITCH_WIDTH)
]

print(f"Start locations outside pitch after adjustment: {len(start_outside)}")
print(f"End locations outside pitch after adjustment: {len(end_outside)}")

Start locations outside pitch after adjustment: 0
End locations outside pitch after adjustment: 0


In [12]:
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)

passes_clean.head()

,counterpress,duration,id,index,location,match_id,minute,off_camera,out,pass_aerial_won,pass_angle,pass_assisted_shot_id,pass_body_part,pass_cross,pass_cut_back,pass_deflected,pass_end_location,pass_goal_assist,pass_height,pass_inswinging,pass_length,pass_miscommunication,pass_no_touch,pass_outcome,pass_outswinging,pass_recipient,pass_recipient_id,pass_shot_assist,pass_straight,pass_switch,pass_technique,pass_through_ball,pass_type,period,play_pattern,player,player_id,position,possession,possession_team,possession_team_id,related_events,second,team,team_id,timestamp,type,under_pressure,start_x,start_y,end_x,end_y
0,None,0.486053,31ad200d-972c-4a06-ad7c-152e65b8578c,5,"[61.0, 40.1]",3825848,0,True,None,None,1.419191,None,Left Foot,None,None,None,"[62.1, 47.3]",None,Ground Pass,None,7.283543,None,None,None,None,Borja González Tomás,6566.0,None,None,None,None,None,Kick Off,1,From Kick Off,Adrián González Morales,6557.0,Center Attacking Midfield,2,Eibar,322,[0fb170d0-078b-4eec-bd0f-93535060bf53],0,Eibar,322,00:00:00.500,Pass,None,61.0,40.1,62.1,47.3
1,None,1.397421,0318204e-f23e-4493-ba61-f1ab39499285,7,"[61.2, 41.3]",3825848,0,True,None,None,-2.137526,None,Right Foot,None,None,None,"[59.1, 38.0]",None,Ground Pass,None,3.911521,None,None,None,None,Daniel García Carrillo,6775.0,None,None,None,None,None,None,1,From Kick Off,Borja González Tomás,6566.0,Center Forward,2,Eibar,322,[03874a60-1451-44b3-9403-ede0dd55dad3],0,Eibar,322,00:00:00.986,Pass,None,61.2,41.3,59.1,38.0
2,None,1.558912,dd445101-ad1b-4de8-8ffc-b5fd3d1f976a,9,"[49.2, 33.5]",3825848,0,None,None,None,2.167094,None,Right Foot,None,None,None,"[32.3, 58.4]",None,Ground Pass,None,30.093521,None,None,None,None,Aleksandar Pantić,27239.0,None,None,None,None,None,None,1,From Kick Off,Daniel García Carrillo,6775.0,Left Defensive Midfield,2,Eibar,322,[a4951807-804c-4166-9328-c636d5beed2e],4,Eibar,322,00:00:04.078,Pass,None,49.2,33.5,32.3,58.4
3,None,1.633776,01a2c6b0-b72b-4771-aa79-fca845a41857,12,"[32.3, 58.4]",3825848,0,None,None,None,-1.630554,None,Right Foot,None,None,None,"[30.2, 23.3]",None,Ground Pass,None,35.162766,None,None,None,None,Mauro Javier Dos Santos,6837.0,None,None,None,None,None,None,1,From Kick Off,Aleksandar Pantić,27239.0,Left Center Back,2,Eibar,322,[43c4d791-3f73-4620-a406-139d56b2af1f],7,Eibar,322,00:00:07.214,Pass,None,32.3,58.4,30.2,23.3
4,None,2.934427,5cd78c17-e82d-454d-b76e-3286edb1936d,15,"[40.5, 76.1]",3825848,0,None,None,None,-1.658705,None,Right Foot,None,None,None,"[37.9, 46.6]",None,Ground Pass,None,29.614355,None,None,None,None,Aleksandar Pantić,27239.0,None,None,None,None,None,None,1,From Kick Off,Mauro Javier Dos Santos,6837.0,Right Center Back,2,Eibar,322,[50ac42fc-bba4-4ac7-9110-2dc620a465bf],13,Eibar,322,00:00:13.855,Pass,None,40.5,76.1,37.9,46.6


# Pass Clustering

In [13]:
from mplsoccer import Pitch

pitch = Pitch(pitch_type='statsbomb', line_color='black')

In [ ]:
# ============================================================
# STATSBOMB-STYLE CLUSTERING + NATURAL DESCRIPTIONS
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import math

# -----------------------------
# SETTINGS
# -----------------------------
CLUSTERS = 50
TOP_CLUSTERS = 5
N_REPRESENTATIVE = 30
SEED = 14

pitch = Pitch(pitch_type='statsbomb', line_color='black')

FEATURES = ["start_x", "start_y", "end_x", "end_y"]

# -----------------------------
# DATA
# -----------------------------
df = passes_clean.copy()
df = df.dropna(subset=FEATURES)

if "length" not in df.columns:
    df["length"] = np.sqrt(
        (df["end_x"] - df["start_x"])**2 +
        (df["end_y"] - df["start_y"])**2
    )

# -----------------------------
# CLUSTERING
# -----------------------------
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

kmeans = KMeans(n_clusters=CLUSTERS, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# -----------------------------
# ✅ TEAM CLUSTER DISTRIBUTION (FIXED)
# -----------------------------
team_cluster_counts = (
    df.groupby(["team", "cluster"])
    .size()
    .reset_index(name="count")
)

team_totals = (
    df.groupby("team")
    .size()
    .reset_index(name="total")
)

team_cluster_dist = team_cluster_counts.merge(team_totals, on="team")
team_cluster_dist["proportion"] = (
    team_cluster_dist["count"] / team_cluster_dist["total"]
)

# League average per cluster
league_avg = (
    team_cluster_dist.groupby("cluster")["proportion"]
    .mean()
    .reset_index(name="league_avg")
)

team_cluster_dist = team_cluster_dist.merge(league_avg, on="cluster")

# Proper z-score (IMPROVED)
league_std = (
    team_cluster_dist.groupby("cluster")["proportion"]
    .std()
    .reset_index(name="league_std")
)

team_cluster_dist = team_cluster_dist.merge(league_std, on="cluster")

team_cluster_dist["z_score"] = (
    (team_cluster_dist["proportion"] - team_cluster_dist["league_avg"]) /
    (team_cluster_dist["league_std"] + 1e-6)   # avoid division by zero
)

# -----------------------------
# REPRESENTATIVE PASSES
# -----------------------------
def get_representative_passes(df_cluster, n=N_REPRESENTATIVE):

    if len(df_cluster) == 0:
        return df_cluster

    X = scaler.transform(df_cluster[FEATURES])
    centroid = X.mean(axis=0)

    dist = np.linalg.norm(X - centroid, axis=1)

    df_cluster = df_cluster.copy()
    df_cluster["dist"] = dist

    return df_cluster.sort_values("dist").head(n)

# -----------------------------
# DESCRIPTION
# -----------------------------
def describe_cluster(cluster_df):

    if len(cluster_df) == 0:
        return "no data"

    avg_len = cluster_df["length"].mean()
    dx = (cluster_df["end_x"] - cluster_df["start_x"]).mean()

    if avg_len < 10:
        length_desc = "short passes"
    elif avg_len < 25:
        length_desc = "medium passes"
    else:
        length_desc = "long passes"

    if dx > 5:
        direction = "moving forward"
    elif dx < -5:
        direction = "moving backward"
    else:
        direction = "played sideways"

    return f"{length_desc}, mostly {direction}"

# -----------------------------
# PLOT TEAM
# -----------------------------
def plot_team(ax, team_name, show_descriptions=True):

    team_df = df[df["team"] == team_name]

    team_clusters = team_cluster_dist[
        team_cluster_dist["team"] == team_name
    ].sort_values("z_score", ascending=False).head(TOP_CLUSTERS)

    cmap = plt.colormaps["tab10"]
    descriptions = []

    for i, row in enumerate(team_clusters.itertuples()):

        cluster_id = row.cluster
        cluster_passes = team_df[team_df["cluster"] == cluster_id]

        reps = get_representative_passes(cluster_passes)
        color = cmap(i % 10)

        for _, p in reps.iterrows():
            ax.arrow(
                p["start_x"], p["start_y"],
                p["end_x"] - p["start_x"],
                p["end_y"] - p["start_y"],
                head_width=1.2,
                head_length=2,
                linewidth=1.3,
                color=color,
                alpha=0.4
            )

        descriptions.append((i + 1, color, describe_cluster(cluster_passes)))

    ax.set_title(team_name, pad=6, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

    y = -0.02

    if show_descriptions:
        for i, (idx, color, desc) in enumerate(descriptions):
            ax.text(
                0.5,
                y - i * 0.06,
                f"➜ {idx}: {desc}",
                transform=ax.transAxes,
                fontsize=9,
                color=color,
                ha="center",
                va="top"
            )
    else:
        x_positions = np.linspace(0.2, 0.8, len(descriptions))
        for x, (idx, color, _) in zip(x_positions, descriptions):
            ax.text(
                x,
                y,
                f"➜ {idx}",
                transform=ax.transAxes,
                fontsize=11,
                color=color,
                ha="center",
                va="top"
            )

# -----------------------------
# GRID
# -----------------------------
def plot_all_teams_grid(n_cols=4, show_descriptions=True):

    teams = sorted(df["team"].dropna().unique())
    n_rows = math.ceil(len(teams) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 4 * n_rows),
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05}
    )

    axes = axes.flatten()

    fig.suptitle(
        "Team Passing Style Clusters | La Liga 2015/16 | Top 5 Clusters (Z-Score vs League)",
        fontsize=16,
        fontweight="bold",
        y=0.91
    )

    for i, team in enumerate(teams):
        pitch.draw(ax=axes[i])
        plot_team(axes[i], team, show_descriptions=show_descriptions)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig("Images and others/team_clusters_laliga1516_allpasses.png", dpi=300, bbox_inches="tight")
    plt.show()

# -----------------------------
# RUN
# -----------------------------
plot_all_teams_grid(n_cols=4, show_descriptions=False)

In [ ]:
# ============================================================
# STATSBOMB-STYLE CLUSTERING (OWN HALF BUILDUP)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import math

# -----------------------------
# SETTINGS
# -----------------------------
CLUSTERS = 50
TOP_CLUSTERS = 5
N_REPRESENTATIVE = 30
SEED = 14

pitch = Pitch(pitch_type='statsbomb', line_color='black')

FEATURES = ["start_x", "start_y", "end_x", "end_y"]

# -----------------------------
# DATA
# -----------------------------
df = passes_clean.copy()
df = df.dropna(subset=FEATURES)

# -----------------------------
# FILTER (OWN HALF BUILDUP)
# -----------------------------
df = df[
    (df["start_x"] <= 60) &
    (df["setpiece"] == 0)
].copy()

# -----------------------------
# LENGTH
# -----------------------------
if "length" not in df.columns:
    df["length"] = np.sqrt(
        (df["end_x"] - df["start_x"])**2 +
        (df["end_y"] - df["start_y"])**2
    )

# -----------------------------
# CLUSTERING
# -----------------------------
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

kmeans = KMeans(n_clusters=CLUSTERS, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# -----------------------------
# DISTRIBUTIONS
# -----------------------------
league_dist = df["cluster"].value_counts(normalize=True)

team_cluster_dist = (
    df.groupby("team")["cluster"]
    .value_counts(normalize=True)
    .rename("team_share")
    .reset_index()
)

team_cluster_dist["league_share"] = team_cluster_dist["cluster"].map(league_dist)

team_cluster_dist["std"] = np.sqrt(
    team_cluster_dist["league_share"] *
    (1 - team_cluster_dist["league_share"])
)

team_cluster_dist["z_score"] = (
    (team_cluster_dist["team_share"] - team_cluster_dist["league_share"])
    / team_cluster_dist["std"]
)

# -----------------------------
# REPRESENTATIVE PASSES
# -----------------------------
def get_representative_passes(df_cluster, n=N_REPRESENTATIVE):

    X = scaler.transform(df_cluster[FEATURES])
    centroid = X.mean(axis=0)

    dist = np.linalg.norm(X - centroid, axis=1)

    df_cluster = df_cluster.copy()
    df_cluster["dist"] = dist

    return df_cluster.sort_values("dist").head(n)

# -----------------------------
# DESCRIPTION
# -----------------------------
def describe_cluster(cluster_df):

    avg_len = cluster_df["length"].mean()
    dx = (cluster_df["end_x"] - cluster_df["start_x"]).mean()

    y_mean = cluster_df["start_y"].mean()
    if y_mean < 20:
        horizontal = "left"
    elif y_mean > 60:
        horizontal = "right"
    else:
        horizontal = "central"

    x_mean = cluster_df["start_x"].mean()
    if x_mean < 30:
        vertical = "deep defensive"
    elif x_mean < 60:
        vertical = "mid defensive"
    else:
        vertical = "advanced"

    if avg_len < 10:
        length_desc = "short passes"
    elif avg_len < 25:
        length_desc = "medium passes"
    else:
        length_desc = "long passes"

    if dx > 5:
        direction = "moving forward"
    elif dx < -5:
        direction = "moving backward"
    else:
        direction = "played sideways"

    return f"{length_desc} from {horizontal} {vertical} areas, mostly {direction}"

# -----------------------------
# PLOT TEAM
# -----------------------------
def plot_team(ax, team_name, show_descriptions=True):

    team_df = df[df["team"] == team_name]

    team_clusters = team_cluster_dist[
        team_cluster_dist["team"] == team_name
    ].sort_values("z_score", ascending=False).head(TOP_CLUSTERS)

    cmap = plt.colormaps["tab10"]

    descriptions = []

    for i, row in enumerate(team_clusters.itertuples()):

        cluster_id = row.cluster
        cluster_passes = team_df[team_df["cluster"] == cluster_id]

        reps = get_representative_passes(cluster_passes)

        color = cmap(i % 10)

        for _, p in reps.iterrows():
            ax.arrow(
                p["start_x"], p["start_y"],
                p["end_x"] - p["start_x"],
                p["end_y"] - p["start_y"],
                head_width=1.2,
                head_length=2,
                linewidth=1.3,
                color=color,
                alpha=0.4
            )

        descriptions.append((i + 1, color, describe_cluster(cluster_passes)))

    ax.set_title(team_name, pad=6, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

    # =========================================================
    # LEGEND (same system as your main version)
    # =========================================================

    y = -0.02

    if show_descriptions:

        for i, (idx, color, desc) in enumerate(descriptions):

            ax.text(
                0.5,
                y - i * 0.06,
                f"➜ {idx}: {desc}",
                transform=ax.transAxes,
                fontsize=9,
                color=color,
                ha="center",
                va="top"
            )

    else:

        x_positions = np.linspace(0.2, 0.8, len(descriptions))

        for x, (idx, color, _) in zip(x_positions, descriptions):

            ax.text(
                x,
                y,
                f"➜ {idx}",
                transform=ax.transAxes,
                fontsize=11,
                color=color,
                ha="center",
                va="top"
            )

# -----------------------------
# GRID PLOT
# -----------------------------
def plot_all_teams_grid(n_cols=4,
                        show_descriptions=False,
                        title="Team Passing Style Clusters | Own Half | Top 5 Clusters (Z-Score vs League)"):

    teams = sorted(df["team"].dropna().unique())
    n_rows = math.ceil(len(teams) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 4 * n_rows),
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05}
    )

    axes = axes.flatten()

    fig.suptitle(
        title,
        fontsize=16,
        fontweight="bold",
        y=0.91
    )

    for i, team in enumerate(teams):

        pitch.draw(ax=axes[i])
        plot_team(axes[i], team, show_descriptions=show_descriptions)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])
    plt.savefig(
        "Images and others/team_clusters_pl1516_ownhalf.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()

# -----------------------------
# RUN
# -----------------------------
plot_all_teams_grid(n_cols=4, show_descriptions=False)

In [ ]:
# ============================================================
# STATSBOMB-STYLE CLUSTERING (DEFENSIVE THIRD BUILDUP)
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mplsoccer import Pitch
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import math

# -----------------------------
# SETTINGS
# -----------------------------
CLUSTERS = 50
TOP_CLUSTERS = 5
N_REPRESENTATIVE = 30
SEED = 14

pitch = Pitch(pitch_type='statsbomb', line_color='black')

FEATURES = ["start_x", "start_y", "end_x", "end_y"]

# -----------------------------
# DATA
# -----------------------------
df = passes_clean.copy()
df = df.dropna(subset=FEATURES)

# -----------------------------
# FILTER (DEFENSIVE THIRD)
# -----------------------------
df = df[
    (df["start_x"] <= 40) &
    (df["setpiece"] == 0)
].copy()

# -----------------------------
# LENGTH
# -----------------------------
if "length" not in df.columns:
    df["length"] = np.sqrt(
        (df["end_x"] - df["start_x"])**2 +
        (df["end_y"] - df["start_y"])**2
    )

# -----------------------------
# CLUSTERING
# -----------------------------
scaler = StandardScaler()
X = scaler.fit_transform(df[FEATURES])

kmeans = KMeans(n_clusters=CLUSTERS, random_state=SEED, n_init=10)
df["cluster"] = kmeans.fit_predict(X)

# -----------------------------
# DISTRIBUTIONS
# -----------------------------
league_dist = df["cluster"].value_counts(normalize=True)

team_cluster_dist = (
    df.groupby("team")["cluster"]
    .value_counts(normalize=True)
    .rename("team_share")
    .reset_index()
)

team_cluster_dist["league_share"] = team_cluster_dist["cluster"].map(league_dist)

team_cluster_dist["std"] = np.sqrt(
    team_cluster_dist["league_share"] *
    (1 - team_cluster_dist["league_share"])
)

team_cluster_dist["z_score"] = (
    (team_cluster_dist["team_share"] - team_cluster_dist["league_share"])
    / team_cluster_dist["std"]
)

# -----------------------------
# REPRESENTATIVE PASSES
# -----------------------------
def get_representative_passes(df_cluster, n=N_REPRESENTATIVE):

    X = scaler.transform(df_cluster[FEATURES])
    centroid = X.mean(axis=0)

    dist = np.linalg.norm(X - centroid, axis=1)

    df_cluster = df_cluster.copy()
    df_cluster["dist"] = dist

    return df_cluster.sort_values("dist").head(n)

# -----------------------------
# DESCRIPTION
# -----------------------------
def describe_cluster(cluster_df):

    avg_len = cluster_df["length"].mean()
    dx = (cluster_df["end_x"] - cluster_df["start_x"]).mean()

    y_mean = cluster_df["start_y"].mean()
    if y_mean < 20:
        horizontal = "left"
    elif y_mean > 60:
        horizontal = "right"
    else:
        horizontal = "central"

    x_mean = cluster_df["start_x"].mean()
    if x_mean < 30:
        vertical = "deep defensive"
    elif x_mean < 60:
        vertical = "mid defensive"
    else:
        vertical = "advanced"

    if avg_len < 10:
        length_desc = "short passes"
    elif avg_len < 25:
        length_desc = "medium passes"
    else:
        length_desc = "long passes"

    if dx > 5:
        direction = "moving forward"
    elif dx < -5:
        direction = "moving backward"
    else:
        direction = "played sideways"

    return f"{length_desc} from {horizontal} {vertical} areas, mostly {direction}"

# -----------------------------
# PLOT TEAM
# -----------------------------
def plot_team(ax, team_name, show_descriptions=True):

    team_df = df[df["team"] == team_name]

    team_clusters = team_cluster_dist[
        team_cluster_dist["team"] == team_name
    ].sort_values("z_score", ascending=False).head(TOP_CLUSTERS)

    cmap = plt.colormaps["tab10"]

    descriptions = []

    for i, row in enumerate(team_clusters.itertuples()):

        cluster_id = row.cluster
        cluster_passes = team_df[team_df["cluster"] == cluster_id]

        reps = get_representative_passes(cluster_passes)

        color = cmap(i % 10)

        for _, p in reps.iterrows():
            ax.arrow(
                p["start_x"], p["start_y"],
                p["end_x"] - p["start_x"],
                p["end_y"] - p["start_y"],
                head_width=1.2,
                head_length=2,
                linewidth=1.3,
                color=color,
                alpha=0.4
            )

        descriptions.append((i + 1, color, describe_cluster(cluster_passes)))

    ax.set_title(team_name, pad=6, fontweight="bold")
    ax.invert_yaxis()
    ax.set_xticks([])
    ax.set_yticks([])

    # =========================================================
    # LEGEND (UNIFIED SYSTEM)
    # =========================================================

    y = -0.02

    if show_descriptions:

        for i, (idx, color, desc) in enumerate(descriptions):

            ax.text(
                0.5,
                y - i * 0.06,
                f"➜ {idx}: {desc}",
                transform=ax.transAxes,
                fontsize=9,
                color=color,
                ha="center",
                va="top"
            )

    else:

        x_positions = np.linspace(0.2, 0.8, len(descriptions))

        for x, (idx, color, _) in zip(x_positions, descriptions):

            ax.text(
                x,
                y,
                f"➜ {idx}",
                transform=ax.transAxes,
                fontsize=11,
                color=color,
                ha="center",
                va="top"
            )

# -----------------------------
# GRID PLOT
# -----------------------------
def plot_all_teams_grid(
    n_cols=4,
    show_descriptions=False,
    title="Team Passing Style Clusters | Defensive Third Build-up | Top 5 Clusters (Z-Score vs League)"
):

    teams = sorted(df["team"].dropna().unique())
    n_rows = math.ceil(len(teams) / n_cols)

    fig, axes = plt.subplots(
        n_rows,
        n_cols,
        figsize=(18, 4 * n_rows),
        gridspec_kw={'hspace': 0.08, 'wspace': 0.05}
    )

    axes = axes.flatten()

    fig.suptitle(
        title,
        fontsize=16,
        fontweight="bold",
        y=0.91
    )

    for i, team in enumerate(teams):

        pitch.draw(ax=axes[i])
        plot_team(axes[i], team, show_descriptions=show_descriptions)

    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout(rect=[0, 0.06, 1, 0.95])

    plt.savefig(
        "Images and others/team_clusters_pl1516_ownthirdbuildup.png",
        dpi=300,
        bbox_inches="tight"
    )

    plt.show()

# -----------------------------
# RUN
# -----------------------------
plot_all_teams_grid(n_cols=4, show_descriptions=False)